In [ ]:
# Install kaggle-benchmarks SDK (and ensure compatible protobuf)
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "kaggle-benchmarks",
        "protobuf>=5.29.6",
    ]
)


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import kaggle_benchmarks as kbench
import pandas as pd
from kaggle_benchmarks import kaggle as _kbk
from kaggle_benchmarks import results as _kb_results

if not _kbk.is_configured():
    raise RuntimeError(
        "Kaggle model proxy is not configured (MODEL_PROXY_URL / MODEL_PROXY_API_KEY missing)."
    )

# The SDK infers result types from __annotations__. With future-annotations
# enabled, task.py exposes the return annotation as the string "float".
_kb_results.types.setdefault("float", _kb_results.Score)

SEARCH_ROOTS = [
    Path.cwd(),
    Path.cwd() / "kaggle",
    Path.cwd().parent,
]


def find_file(name: str) -> Path:
    candidates: list[Path] = []
    for root in SEARCH_ROOTS:
        candidates.append(root / name)
        candidates.append(root / "kaggle" / name)
    for candidate in candidates:
        if candidate.exists():
            return candidate
    checked = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Could not find {name!r}. Checked:\n{checked}")


task_path = find_file("task.py")
questions_path = find_file("questions.jsonl")
artifact_dir = Path.cwd() / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

if str(task_path.parent) not in sys.path:
    sys.path.insert(0, str(task_path.parent))

import task

df = pd.read_json(questions_path, lines=True)
df = df.rename(columns={"class": "class_"})

print(f"Loaded {len(df)} rows from {questions_path}")
print(df[["id", "class_", "difficulty", "seed"]].to_string(index=False))
print(f"Task path: {task_path}")
print(f"Registered task: {getattr(task.run, 'name', 'run')}")
print(f"Artifact dir: {artifact_dir}")


In [ ]:
detail_rows: list[dict[str, object]] = []
detail_dir = artifact_dir / "detail_rows"
detail_dir.mkdir(parents=True, exist_ok=True)


def detailed_run(
    llm,
    id,
    class_,
    difficulty,
    seed,
    instance,
    gold_objective,
    baseline_objective,
    value_cap,
    wall_budget_s=1800,
    components=None,
):
    del wall_budget_s

    inst = task._coerce_json(instance, default=None)
    if not isinstance(inst, dict):
        raise TypeError("instance must be a dict or JSON object string")

    row = task.run_instance(
        llm=llm,
        instance=inst,
        cls=str(class_),
        difficulty=str(difficulty),
        seed=int(seed),
        gold_objective=float(gold_objective),
        baseline_objective=float(baseline_objective),
        value_cap=float(value_cap),
        components=task._normalize_components(components),
    )

    detail = {
        "id": str(id),
        "class": str(class_),
        "difficulty": str(difficulty),
        "seed": int(seed),
        "score": float(row["score"]),
        "score_at_stop": float(row["score_at_stop"]),
        "score_after_cf": float(row["score_after_cf"]),
        "cf_delta": float(row["cf_delta"]),
        "stop_reason": row["stop_reason"],
        "wall_s": float(row["wall_s"]),
        "n_exec_turns": int(row["n_exec_turns"]),
        "error": row["error"],
        "parsed": row["parsed"],
        "cf_parsed": row["cf_parsed"],
        "transcript": row["transcript"],
    }
    detail_rows.append(detail)
    (detail_dir / f"{detail['id']}.json").write_text(
        json.dumps(detail, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    return float(row["score"])


if not hasattr(task.run, "func"):
    raise TypeError("task.run is not a mutable kbench Task object; cannot instrument evaluate().")

original_func = task.run.func
object.__setattr__(task.run, "func", detailed_run)
try:
    runs = task.run.evaluate(
        llm=[kbench.llm],
        evaluation_data=df,
        timeout=1800,
        n_jobs=1,
    )
finally:
    object.__setattr__(task.run, "func", original_func)

eval_df = runs.as_dataframe().reset_index()
eval_summary = eval_df[
    [column for column in ["id", "class_", "difficulty", "seed", "result"] if column in eval_df.columns]
].copy()
eval_summary = eval_summary.rename(columns={"class_": "class"})

summary_df = pd.DataFrame(detail_rows).sort_values("id").reset_index(drop=True)
missing_ids = sorted(set(df["id"]) - set(summary_df["id"]))

(artifact_dir / "detailed_rows.json").write_text(
    json.dumps(detail_rows, indent=2),
    encoding="utf-8",
)
summary_df.to_csv(artifact_dir / "detailed_rows.csv", index=False)
(artifact_dir / "evaluate_runs.json").write_text(
    json.dumps(eval_summary.to_dict(orient="records"), indent=2),
    encoding="utf-8",
)
(artifact_dir / "missing_ids.json").write_text(
    json.dumps(missing_ids, indent=2),
    encoding="utf-8",
)

print("Evaluate summary:")
print(eval_summary.to_string(index=False))

summary_cols = [
    "id",
    "class",
    "score",
    "score_at_stop",
    "score_after_cf",
    "cf_delta",
    "stop_reason",
    "wall_s",
    "error",
]
print("\nDetailed per-row summary:")
print(summary_df[summary_cols].to_string(index=False))

decision_df = summary_df[
    (summary_df["stop_reason"] == "decision_stop") & summary_df["cf_delta"].notna()
]

print(f"\nDecision-stop rows: {len(decision_df)}")
if not decision_df.empty:
    print(
        decision_df[
            ["id", "class", "cf_delta", "score_at_stop", "score_after_cf"]
        ].to_string(index=False)
    )

assert len(summary_df) == len(df), (
    f"Expected {len(df)} rows, captured {len(summary_df)}. Missing: {missing_ids}"
)
assert not decision_df.empty, "No row produced stop_reason='decision_stop' with cf_delta populated."
print(f"Artifacts written to {artifact_dir}")
